In [1]:
import os, sys
import networkx as nx
import plotly.graph_objects as go
from typing import Any

In [ ]:
sys.path.append(os.path.join(os.getcwd(), 'structures'))
from structures.graph import Graph
from structures.problem import SearchResult
from structures.problems.routing import RoutingProblem, RoutingState
from structures.algorithms.blind_search import bfs, dfs, ucs, iddfs, bidirectional_search
from structures.algorithms.heuristic_search import greedy_best_first_search, a_star_search

In [3]:
def print_result(name: str, res: SearchResult[Any, Any], show_solution: bool = True) -> None:
    print(f"\n== {name} ==")
    print(f"found: {res.found}")
    print(f"expanded: {res.expanded} | generated: {res.generated} | max_frontier: {res.max_frontier} | elapsed_ms: {res.elapsed_ms:.2f}")
    if res.found and show_solution:
        print(f"steps: {len(res.actions)} | path_cost: {res.path_cost}")
        # mostra assignments se houver
        if hasattr(res.state, "as_dict") and res.state is not None:
            A = res.state.as_dict()
            print("assignment:", A)

In [4]:
def build_romania_map_graph() -> Graph:
    g = Graph(directed=False)
    edges = [
        ("Arad", "Zerind", 75), ("Arad", "Sibiu", 140), ("Arad", "Timisoara", 118),
        ("Zerind", "Oradea", 71),
        ("Oradea", "Sibiu", 151),
        ("Sibiu", "Fagaras", 99), ("Sibiu", "Rimnicu Vilcea", 80),
        ("Fagaras", "Bucharest", 211),
        ("Rimnicu Vilcea", "Pitesti", 97), ("Rimnicu Vilcea", "Craiova", 146),
        ("Timisoara", "Lugoj", 111),
        ("Lugoj", "Mehadia", 70),
        ("Mehadia", "Drobeta", 75),
        ("Drobeta", "Craiova", 120),
        ("Pitesti", "Bucharest", 101),
        ("Bucharest", "Giurgiu", 90),
        ("Bucharest", "Urziceni", 85),
        ("Urziceni", "Hirsova", 98),
        ("Hirsova", "Eforie", 86),
        ("Urziceni", "Vaslui", 142),
        ("Vaslui", "Iasi", 92),
        ("Iasi", "Neamt", 87)
    ]

    for (u, v, w) in edges:
        g.add_edge(u, v, weight=w)

    return g

In [5]:
g = build_romania_map_graph()

problem = RoutingProblem(g, "Arad", "Bucharest")

In [6]:
dfs_res = dfs(problem)
bfs_res = bfs(problem)
ucs_res = ucs(problem)
iddfs_res = iddfs(problem, max_depth=20)
bidirectional_res = bidirectional_search(problem, RoutingState("Bucharest"))

In [7]:
print_result("DFS Results", dfs_res)


== DFS Results ==
found: True
expanded: 8 | generated: 11 | max_frontier: 3 | elapsed_ms: 0.12
steps: 8 | path_cost: 838.0


In [8]:
dfs_res

SearchResult(found=True, state=RoutingState(location='Bucharest'), actions=[move(Arad -> Timisoara), move(Timisoara -> Lugoj), move(Lugoj -> Mehadia), move(Mehadia -> Drobeta), move(Drobeta -> Craiova), move(Craiova -> Rimnicu Vilcea), move(Rimnicu Vilcea -> Pitesti), move(Pitesti -> Bucharest)], path_cost=838.0, expanded=8, generated=11, max_frontier=3, elapsed_ms=0.11549999908311293)

In [9]:
print_result("Bidirectional Results", bidirectional_res)


== Bidirectional Results ==
found: True
expanded: 4 | generated: 11 | max_frontier: 7 | elapsed_ms: 0.07
steps: 3 | path_cost: nan


In [10]:
bidirectional_res

SearchResult(found=True, state=RoutingState(location='Fagaras'), actions=[move(Arad -> Sibiu), move(Sibiu -> Fagaras), move(Bucharest -> Fagaras)], path_cost=nan, expanded=4, generated=11, max_frontier=7, elapsed_ms=0.06797999958507717)

In [11]:
def heuristic_sld(state: RoutingState) -> float:
    """Heuristic function: straight-line distance to Bucharest."""
    sld = {
        "Arad": 366,
        "Bucharest": 0,
        "Craiova": 160,
        "Drobeta": 242,
        "Eforie": 161,
        "Fagaras": 176,
        "Giurgiu": 77,
        "Hirsova": 151,
        "Iasi": 226,
        "Lugoj": 244,
        "Mehadia": 241,
        "Neamt": 234,
        "Oradea": 380,
        "Pitesti": 100,
        "Rimnicu Vilcea": 193,
        "Sibiu": 253,
        "Timisoara": 329,
        "Urziceni": 80,
        "Vaslui": 199,
        "Zerind": 374
    }
    return sld.get(state.location, float('inf'))

In [12]:
greedy_best_first_search_res = greedy_best_first_search(problem, heuristic_sld)
greedy_best_first_search_res

SearchResult(found=True, state=RoutingState(location='Bucharest'), actions=[move(Arad -> Sibiu), move(Sibiu -> Fagaras), move(Fagaras -> Bucharest)], path_cost=450.0, expanded=4, generated=8, max_frontier=5, elapsed_ms=0.055110000175773166)

In [13]:
a_star_search_res = a_star_search(problem, heuristic_sld)
a_star_search_res

SearchResult(found=True, state=RoutingState(location='Bucharest'), actions=[move(Arad -> Sibiu), move(Sibiu -> Rimnicu Vilcea), move(Rimnicu Vilcea -> Pitesti), move(Pitesti -> Bucharest)], path_cost=418.0, expanded=6, generated=11, max_frontier=6, elapsed_ms=0.0730399988242425)

In [15]:
greedy_best_first_search_res

SearchResult(found=True, state=RoutingState(location='Bucharest'), actions=[move(Arad -> Sibiu), move(Sibiu -> Fagaras), move(Fagaras -> Bucharest)], path_cost=450.0, expanded=4, generated=8, max_frontier=5, elapsed_ms=0.06555999971169513)

In [16]:
networkx_g = g.to_networkx()
pos = nx.spring_layout(networkx_g, seed=42)

fig = go.Figure()
for u, v, data in networkx_g.edges(data=True):
    fig.add_trace(go.Scatter(x=[pos[u][0], pos[v][0]], y=[pos[u][1], pos[v][1]], mode='lines', line=dict(color='blue', width=2)))
    mid_x = (pos[u][0] + pos[v][0]) / 2
    mid_y = (pos[u][1] + pos[v][1]) / 2
    fig.add_trace(go.Scatter(x=[mid_x], y=[mid_y], mode='text', text=str(data['weight']), textposition='top center', showlegend=False))
for node, (x, y) in pos.items():
    fig.add_trace(go.Scatter(x=[x], y=[y], mode='markers+text', text=node, textposition='top center', marker=dict(color='red', size=10)))
fig.update_layout(title="Romania Map Graph", showlegend=False)
fig.show()